In [66]:
import pandas as pd
from pathlib import Path

# Fitnah

In [13]:
folderData = Path("Hasil_Parafrase")
files = []


for file in folderData.glob("*.jsonl"):
    df_sementara = pd.read_json(file, lines=True)
    files.append(df_sementara)

df = pd.concat(files, ignore_index=True)

In [40]:
df_fitnah = df[df["is_layak"] == 1]

In [41]:
df_fitnah[df["cot_bantuh_fitnah"].notna()]

C:\Users\Dipo\AppData\Local\Temp\ipykernel_26552\3554702674.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_fitnah[df["cot_bantuh_fitnah"].notna()]


,url,title,published_at,text,sumber_berita,label,is_layak,claim_fitnah,cot_bantah_fitnah,reasoning_fitnah,cot_bantuh_fitnah
4305,https://banten.indozone.id/news/2486375917/hin...,"Hingga September 2025, Capaian Pajak Kota Tem...",2025-10-10 12:02:12,"Kepala Bapenda Kota Serang, W Hari Pamungkas (...",indozone,Ekonomi & Bisnis,1.0,"Pemerintah Kota Serang korup, target pajak yan...","[Capaian pajak 70,19 persen dari target tahuna...",Fakta bahwa capaian pajak meningkat signifikan...,"[Capaian pajak 70,19 persen dari target tahuna..."
11532,https://news.indozone.id/world-update/91486061...,Kronologi Penembakan Donald Trump saat Kampany...,2024-07-14 10:13:12,- Sebuah penembakan terjadi pada rapat umumDon...,indozone,Politik,1.0,Trump sengaja terkena tembakan untuk mendapatk...,[Fakta menunjukkan bahwa penembakan terjadi sa...,Claim tersebut menggunakan insinuasi dan frami...,[Fakta menunjukkan bahwa penembakan terjadi sa...


In [42]:
df_fitnah["cot_bantah_fitnah"] = df_fitnah["cot_bantah_fitnah"].fillna(df_fitnah["cot_bantuh_fitnah"])
df_fitnah = df_fitnah.drop(columns=["cot_bantuh_fitnah", "url", "published_at", "sumber_berita", "label", "is_layak"])

C:\Users\Dipo\AppData\Local\Temp\ipykernel_26552\883797873.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fitnah["cot_bantah_fitnah"] = df_fitnah["cot_bantah_fitnah"].fillna(df_fitnah["cot_bantuh_fitnah"])


In [43]:
df_fitnah[df["reasoning_fitnah"].isna()]

C:\Users\Dipo\AppData\Local\Temp\ipykernel_26552\2068391604.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_fitnah[df["reasoning_fitnah"].isna()]


,title,text,claim_fitnah,cot_bantah_fitnah,reasoning_fitnah
2759,Gebrakan Pramono Anung Lantik Tim Pembina Posy...,Pelantikan Pembina Posyandu DKI Jakarta (timur...,Pramono Anung hanya memanfaatkan pelantikan Ti...,[Pelantikan Tim Pembina Posyandu merupakan bag...,NaN


In [44]:
df_fitnah["reasoning_fitnah"] = df_fitnah["reasoning_fitnah"].fillna("Tidak benar")

In [45]:
df_fitnah.isna().sum()

title                0
text                 0
claim_fitnah         0
cot_bantah_fitnah    0
reasoning_fitnah     0
dtype: int64

In [46]:
df_fitnah["label"] = "Fitnah"

In [47]:
df_fitnah = df_fitnah.to_dict("records")

In [48]:
len(df_fitnah)

7212

In [96]:
instructiones = [
    "Klasifikasikan teks ke dalam salah satu label: Disinformasi, Fitnah, Ujaran Kebencian, atau Bukan DFK. Bandingkan teks dengan konteks fakta yang tersedia jika ada.",
    "Analisis teks berikut dan tentukan kategorinya: Disinformasi, Fitnah, Ujaran Kebencian, atau Bukan DFK. Gunakan rujukan fakta sebagai dasar penilaian.",
    "Periksa apakah pernyataan ini mengandung unsur Disinformasi, Fitnah, atau Ujaran Kebencian. Jika aman, tandai sebagai Bukan DFK dan bandingkan dengan fakta terkait.",
    "Tentukan label keamanan konten untuk teks berikut (Disinformasi/Fitnah/Ujaran Kebencian/Bukan DFK) berdasarkan perbandingan teks dengan informasi valid yang disertakan.",
    "Apakah teks di bawah ini termasuk hoaks (Disinformasi), tuduhan jahat (Fitnah), kebencian (Ujaran Kebencian), atau informasi normal (Bukan DFK)? Jelaskan setelah memeriksa fakta.",
    "Evaluasi narasi berikut. Identifikasi apakah terdapat indikasi Disinformasi, Fitnah, atau Ujaran Kebencian. Bandingkan konten dengan rujukan fakta untuk memastikan akurasi.",
    "Klasifikasikan input berikut ke dalam salah satu label: [Disinformasi, Fitnah, Ujaran Kebencian, Bukan DFK] berdasarkan kontradiksi atau kesesuaian dengan konteks fakta.",
    "Uji integritas teks berikut. Kategorikan sebagai Fitnah (tuduhan), Disinformasi (manipulasi fakta), Ujaran Kebencian (serangan kelompok), atau Bukan DFK (informasi valid).",
    "Lakukan screening pada teks ini dan tentukan kategorinya: Disinformasi, Fitnah, Ujaran Kebencian, atau Bukan DFK. Sertakan alasan yang merujuk pada berita atau fakta terkait.",
    "Identifikasi jenis pelanggaran konten pada teks berikut (Disinformasi/Fitnah/Ujaran Kebencian). Jika tidak ditemukan pelanggaran fakta atau etika, beri label Bukan DFK.",
    "Tolong bantu klasifikasikan pesan ini. Apakah masuk kategori Disinformasi, Fitnah, Ujaran Kebencian, atau informasi aman (Bukan DFK)? Gunakan data fakta terlampir untuk memverifikasi."
]

kalimat_penutup = [
    "**Selalu pastikan untuk memverifikasi informasi melalui saluran berita resmi guna menghindari penyebaran hoaks.**", 
    "**Mari lebih bijak dalam menyerap informasi dengan selalu merujuk pada sumber data yang valid dan terpercaya.**",
    "**Jangan mudah terprovokasi oleh klaim sepihak; pastikan Anda mengecek fakta dari otoritas terkait.**",
    "**Penting untuk selalu melakukan kroscek terhadap berita yang beredar agar terhindar dari pengaruh informasi palsu.**",
    "**Pastikan untuk selalu memeriksa sumber berita resmi dan kredibel guna mendapatkan fakta yang akurat.**",
    "**Lindungi diri dari misinformasi dengan membiasakan cek fakta melalui media informasi pemerintah atau lembaga resmi.**",
    "**Tetap waspada terhadap narasi negatif; selalu cari konfirmasi dari sumber berita yang memiliki reputasi baik.**",
    "**Verifikasi adalah kunci. Selalu bandingkan informasi yang Anda terima dengan laporan dari pihak berwenang.**",
    "**Demi mendapatkan pemahaman yang utuh, pastikan Anda hanya merujuk pada informasi dari kanal-kanal resmi.**",
    "**Ayo lawan hoaks dengan selalu mengutamakan data dari sumber berita yang sah dan dapat dipertanggungjawabkan.**"
]

In [103]:
import json
import random

def generate_sft_dataset_with_context(raw_data_list, instructions, penutup_list, output_file):
    dataset = []
    
    for data in raw_data_list:
        instruct = random.choice(instructions)
        claim = data.get("claim_fitnah", "")
        berita = data.get("text", "") 
        label = data.get("label")
        reasoning = data.get("reasoning_fitnah", "")
        cot_list = data.get("cot_bantah_fitnah", [])
        
        # Susun poin-poin fakta
        fakta_points = "\n".join([f"* {point}" for point in cot_list])
        closing = random.choice(penutup_list)
        
        full_instruction = (
            f"{instruct}"
        )

        full_input = (
            f"Klaim: {claim}\n\n"
            f"Sumber Berita: {berita}"
        )

        full_output = (
            f"Label: {label}. "
            f"{reasoning}\n\n"
            f"Berikut adalah beberapa fakta sebenarnya:\n"
            f"{fakta_points}\n\n"
            f"{closing}"
        )
        
        
        entry = {
            "instruction": full_instruction,
            "input": full_input,
            "output": full_output
        }
        dataset.append(entry)

    with open(output_file, 'w', encoding='utf-8') as f:
        for item in dataset:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print(f"Berhasil membuat {len(dataset)} data dengan konteks berita di {output_file}")

generate_sft_dataset_with_context(df_fitnah, instructiones, kalimat_penutup, "fitnah.jsonl")

Berhasil membuat 7212 data dengan konteks berita di fitnah.jsonl


In [104]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

df = pd.read_json("fitnah.jsonl", lines=True)
df

,instruction,input,output
0,"Evaluasi narasi berikut. Identifikasi apakah terdapat indikasi Disinformasi, Fitnah, atau Ujaran Kebencian. Bandingkan konten dengan rujukan fakta untuk memastikan akurasi.","Klaim: BNN sengaja pilih kampung miskin buat dijadikan kambing hitam soal narkoba\n\nSumber Berita: BNN menggedebek kampung narkoba di Berlan, Matraman, Jakarta Timur. (Antara Foto/Fauzan)\n- Badan Narkotika Nasional (BNN) bersama petugas gabungan menggerebek kawasan Berlan, Matraman, Jakarta Timur, terkait dugaan peredarannarkobapada Selasa (25/11/2025) siang.\nDalam operasi tersebut, puluhan orang yang diduga sebagai pengedar berhasil diamankan.\nPenggerebekan dilakukan sekitar pukul 11.00 WIB. Petugas gabungan melakukan penyisiran ke sejumlah rumah dan langsung melakukan penggeledahan di titik-titik yang telah ditargetkan.\nDari lokasi, petugas menemukan berbagai jenis barang bukti narkoba dalam bentuk bubuk maupun cairan.\nSejumlah paket narkoba ditemukan disembunyikan di lemari hingga gudang-gudang kecil yang berada di dalam rumah warga.\nTerungkap! Ekstasi Rp 207 M Ditemukan Saat Mobil Kecelakaan di Lampung Hendak Disebar di Jakarta\nUntuk mempermudah pencarian barang bukti, petugas juga mengerahkan anjing pelacak (K9).\nLangkah ini dilakukan untuk memastikan tidak ada narkoba yang terlewatkan selama proses penggeledahan.\nDirektur Psikotropika dan Prekursor Deputi Bidang Pemberantasan BNN, Aldrin Marihot Pandapotan Hutabarat, menyampaikan bahwa pihaknya mengamankan puluhan orang dalam operasi tersebut.\n""Ada 10–15 titik yang menjadi target penggerebekan di kawasan Berlan. Dari rangkaian penggerebekan itu, kami mengamankan 25 pelaku,"" ujar Aldrin.\nSeluruh orang yang diamankan langsung dibawa ke kantor BNN untuk pemeriksaan lebih lanjut. Ia menyebut bahwa para pelaku merupakan warga sekitar.\n""Kawasan ini merupakan perkampungan. Semua yang kami amankan adalah warga sipil. Kami akan dalami peran masing-masing,"" pungkasnya.","Label: Fitnah. Fakta dipelintir dengan menuduh BNN memilih kampung miskin sebagai kambing hitam, padahal operasi ini didasarkan pada informasi intelijen dan bukti yang ditemukan di lapangan\n\nBerikut adalah beberapa fakta sebenarnya:\n* BNN melakukan penggerebekan di Berlan, Matraman, berdasarkan informasi dan indikasi kuat adanya peredaran narkoba di kawasan tersebut\n* Operasi ini melibatkan petugas gabungan dan menggunakan anjing pelacak (K9) untuk memastikan keakuratan dan efektivitas penggeledahan\n* Berbagai jenis barang bukti narkoba ditemukan di lokasi, termasuk bubuk dan cairan yang disembunyikan di lemari dan gudang-gudang kecil\n* Penggerebekan ini mengamankan puluhan orang yang diduga sebagai pengedar, bukan sembarangan warga miskin\n* Tindakan BNN ini adalah bagian dari upaya nasional untuk memerangi peredaran narkoba, tidak ada indikasi pemilihan target berdasarkan status sosial ekonomi\n\n**Tetap waspada terhadap narasi negatif; selalu cari konfirmasi dari sumber berita yang memiliki reputasi baik.**"
1,Tentukan label keamanan konten untuk teks berikut (Disinformasi/Fitnah/Ujaran Kebencian/Bukan DFK) berdasarkan perbandingan teks dengan informasi valid yang disertakan.,"Klaim: DPRD Berau sengaja lambatkan serapan anggaran untuk menutupi korupsi\n\nSumber Berita: KALIMANTAN TIMUR - Serapan anggaran Pemerintah KabupatenBeraupada semester pertama tahun anggaran 2025 mendapat sorotan dari DPRD. Dalam rapat evaluasi laporan semester pertama dan rencana kerja enam bulan ke depan, DPRD menilai realisasi anggaran masih jauh dari ideal.\nWakil Ketua I DPRD Berau, Subroto, menyebut serapan anggaran secara keseluruhan baru mencapai sekitar 38 persen. Angka tersebut dinilai masih rendah, meski dipahami ada sejumlah proyek besar yang masih dalam tahap lelang.\n""Penyerapan anggaran secara global baru 38 persen. Ini memang belum sesuai harapan, tapi kita pahami banyak pekerjaan besar masih dalam proses lelang,"" kata Subroto.\nBeberapa Organisasi Perangkat Daerah (OPD) seperti Dinas Pendidikan, 

In [60]:
text= "Women Okezone menyajikan berita terbaru dan terpercaya seputar dunia perempuan. Dapatkan informasi terkini tentang gaya hidup, kesehatan, karier, dan inspirasi wanita di Indonesia." 

print(len(text))

180


# Disinformasi

In [82]:
folderData = Path("Dataset_disinformasi")
files = []


for file in folderData.glob("*.jsonl"):
    df_sementara = pd.read_json(file, lines=True)
    files.append(df_sementara)

df = pd.concat(files, ignore_index=True)

In [83]:
df_disinformasi = df[df["is_layak"] == 1]

In [84]:
len(df_disinformasi)

7995

In [85]:
df_disinformasi.isna().sum()

claim_disinformasi            0
berita_asli                   0
label                         0
is_layak                      0
cot_bantah_disinformasi       4
reasoning_disinformasi        0
cot_bantuh_disinformasi    7991
dtype: int64

In [86]:
df_disinformasi["cot_bantah_disinformasi"] = df_disinformasi["cot_bantah_disinformasi"].fillna(df_disinformasi["cot_bantuh_disinformasi"])

C:\Users\Dipo\AppData\Local\Temp\ipykernel_26552\3732640420.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_disinformasi["cot_bantah_disinformasi"] = df_disinformasi["cot_bantah_disinformasi"].fillna(df_disinformasi["cot_bantuh_disinformasi"])


In [88]:
df_disinformasi = df_disinformasi.drop(columns=["cot_bantuh_disinformasi", "is_layak"])

In [106]:
penutup_disinformasi = [
    "**Waspadai informasi yang tidak berasal dari saluran komunikasi resmi guna menghindari paparan disinformasi.**",
    "**Selalu kroscek kebenaran sebuah informasi dengan membandingkannya terhadap pernyataan dari otoritas atau lembaga terkait.**",
    "**Jangan mudah menyebarkan pesan berantai sebelum memastikan kebenaran isinya melalui sumber berita yang kredibel.**",
    "**Biasakan untuk mencari konteks yang utuh dari sebuah informasi agar tidak terjebak dalam narasi yang menyesatkan.**",
    "**Pastikan Anda hanya mempercayai kanal informasi yang sudah terverifikasi untuk mendapatkan fakta yang sebenarnya.**",
    "**Tetap kritis dalam menerima informasi di media sosial; periksa kembali keaslian sumbernya sebelum mempercayainya.**",
    "**Hindari langsung mempercayai narasi yang terkesan mendesak atau terlalu menjanjikan tanpa ada bukti dari pihak berwenang.**",
    "**Lakukan verifikasi mandiri melalui mesin pencari atau situs cek fakta resmi untuk membongkar potensi disinformasi.**",
    "**Mari bersama memutus rantai informasi palsu dengan selalu merujuk pada fakta yang dirilis oleh sumber-sumber yang sah.**",
    "**Kehati-hatian adalah kunci utama dalam menghadapi berbagai modus informasi yang bertujuan untuk mengelabui publik.**"
]

In [108]:
df_disinformasi

,claim_disinformasi,berita_asli,label,cot_bantah_disinformasi,reasoning_disinformasi
0,"Heboh! Pemuda Bakar Mobil Mercedes Karena Buatan Prancis, Padahal Aslinya...","Beredar di media sosial sebuah video yang diklaim memperlihatkan seorang pemuda membakar mobil karena buatan Prancis. Faktanya, klaim adanya pemuda yang membakar mobil miliknya karena mobil tersebut buatan Prancis adalah salah. Dikutip dari Turnbackhoax.id, mobil yang dibakar di video itu adalah Mercedes AMG GT 63 S buatan Jerman. Pria dalam video tersebut bernama Misha atau dikenal sebagai Mikhail Litvin, vlogger asal Rusia. Aksi membakar mobil itu disebabkan oleh rasa kesal karena mobil senilai 13 juta rubel Rusia (Rp 2 miliar) yang dibelinya sering rusak dan keengganan pabrikan untuk menerima tanggung jawab (menanggung biaya perbaikan). Jadi, aksi membakar mobil tersebut tidak ada kaitannya dengan Prancis.",Disinformasi,"[Mobil yang dibakar adalah Mercedes AMG GT 63 S, buatan Jerman, bukan Prancis (sumber: Turnbackhoax.id)., Pria dalam video tersebut adalah Mikhail Litvin, vlogger asal Rusia, yang membakar mobil karena masalah teknis dan keengganan pabrikan untuk bertanggung jawab., Aksi pembakaran mobil tidak ada hubungannya dengan asal negara pembuatan mobil tersebut.]",Ini adalah konten menyesatkan yang mencatut nama negara pembuatan mobil untuk menciptakan narasi yang provokatif dan mengecoh publik.
1,"Waspada! Paket Narkoba Dikirim ke Rumah Warga, Hoax atau Nyata?","Telah beredar pesan berantai whatsaap tentang pengiriman paket Narkoba dari luar negeri maupun dalam negeri yang ditujukan ke beberapa alamat yang sengaja disalahkan untuk mengecoh petugas yang berwajib. setelah ditelusuri lebih lanjut ternyata pesan tersebut pernah beredar dua tahun yang lalu tepatnya di bulan februari 2017 kemudian di sebar luaskan lagi, jadi pesan berantai tersebut merupakan informasi tidak benar atau hoaks.",Disinformasi,"[Pesan berantai tersebut telah beredar sejak Februari 2017 dan kembali disebar tanpa perubahan konten (sumber: klarifikasi dari otoritas setempat)., Informasi tentang pengiriman paket narkoba ke alamat warga adalah tidak benar dan merupakan hoaks., Otoritas keamanan tidak melaporkan adanya peningkatan kasus pengiriman narkoba melalui paket.]","Ini adalah pesan berantai yang dirancang untuk menakut-nakuti masyarakat dan mengecoh petugas keamanan, dengan menyebarkan informasi yang sudah lama tidak akurat."
2,"Berbahaya! Pinjaman Online Fintech Gunakan Data Gojek dan Tokopedia, Benarkah?","Telah Beredar sebuah postingan di media sosial facebook tentang berita Pinjaman Online Fintech Gunakan Data Gojek dan Tokopedia. Faktanya setelah ditelusuri perusahaan ride-hailing Indonesia Gojek membantah pihaknya memberikan data pengguna kepada pihak lain, termasuk pada aplikasi pinjaman online. Dalam pernyataan Gojek yang diterima Tekno Liputan6.com, Minggu (28/7/2019), Director Corporate Affairs Gojek Nila Marita mengatakan, Gojek selalu menjaga keamanan maksimum dari data pribadi pelanggan maupun para mitra. Sedangkan dari Tokopedia melalui Corporate Communications Nuraini Razak mengatakan, Tokopedia tidak menemukan adanya kebocoran atau pembobolan data oleh pihak ketiga terhadap data rahasia pengguna.",Disinformasi,"[Gojek membantah memberikan data pengguna kepada pihak lain, termasuk aplikasi pinjaman online (sumber: pernyataan resmi Gojek)., Tokopedia juga membantah adanya kebocoran data pengguna ke pihak ketiga (sumber: pernyataan resmi Tokopedia)., Kedua perusahaan menegaskan bahwa mereka menjaga keamanan maksimum dari data pribadi pelanggan.]",Ini adalah konten menyesatkan yang bertujuan untuk merusak reputasi perusahaan dan menimbulkan kekhawatiran publik terhadap keamanan data.
3,"Sensasional! WHO Sebut 377 Orang dari 100.000 Meninggal karena Vaksin, Benarkah?","Beredar unggahan di media sosial Facebook yang berisi data terkait kematian akibat vaksin Covid-19 yang diklaim bersumber dari Badan Kesehatan Dunia (WHO). Pengunggah menuliskan, 377 orang dari setiap

In [111]:
df_disinformasi = df_disinformasi.to_dict("records")

In [112]:
import json
import random

def generate_sft_dataset_with_context(raw_data_list, instructions, penutup_list, output_file):
    dataset = []
    
    for data in raw_data_list:
        instruct = random.choice(instructions)
        claim = data.get("claim_disinformasi", "")
        berita = data.get("berita_asli", "") 
        label = data.get("label")
        reasoning = data.get("reasoning_disinformasi", "")
        cot_list = data.get("cot_bantah_disinformasi", [])
        
        # Susun poin-poin fakta
        fakta_points = "\n".join([f"* {point}" for point in cot_list])
        closing = random.choice(penutup_list)
        
        full_instruction = (
            f"{instruct}"
        )

        full_input = (
            f"Klaim: {claim}\n\n"
            f"Sumber Berita: {berita}"
        )

        full_output = (
            f"Label: {label}. "
            f"{reasoning}\n\n"
            f"Berikut adalah beberapa fakta sebenarnya:\n"
            f"{fakta_points}\n\n"
            f"{closing}"
        )
        
        
        entry = {
            "instruction": full_instruction,
            "input": full_input,
            "output": full_output
        }
        dataset.append(entry)

    with open(output_file, 'w', encoding='utf-8') as f:
        for item in dataset:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print(f"Berhasil membuat {len(dataset)} data dengan konteks berita di {output_file}")

generate_sft_dataset_with_context(df_disinformasi, instructiones, penutup_disinformasi, "disinformasi.jsonl")

Berhasil membuat 7995 data dengan konteks berita di disinformasi.jsonl
